# 03 · Surfability rules and the condition-fit score

**Purpose.** For a spot, a skill level, a time window and a snapshot, compute:
- a **condition-fit score** `CF ∈ [0, 1]` (how well forecast conditions suit this spot and this surfer),
- a **surfable / not surfable** call (`CF ≥ surfable_threshold`),
- the **best hour**, the hours that are surfable, and
- a short **English explanation** built only from the computed numbers.

**Inputs:** a snapshot from `data/snapshots/` (00), `data/processed/spots_clean.json` (01), `data/resources/surfability_thresholds.csv` (all thresholds, editable), optional `data/resources/cwa_stations_en.csv`

**Outputs:** functions used by `04_search` and `05_evaluation`. When run directly it also writes `results/figures/surfability_heatmap.png` and `results/surfability_table.csv`.

> **[TEAM] The shipped threshold values are PLACEHOLDERS** chosen only so the code runs. Replace each value with one from your cited source or surf-school interview, and record the source in the `source` column. The code never hard-codes a threshold.

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
SNAP_ROOT = DATA_DIR / "snapshots"
CORPUS_FILE = DATA_DIR / "processed" / "spots_clean.json"
THRESHOLDS_FILE = DATA_DIR / "resources" / "surfability_thresholds.csv"
STATION_LOOKUP_FILE = DATA_DIR / "resources" / "cwa_stations_en.csv"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"

LIBRARY_MODE = os.environ.get("SURF_IR_LIBRARY_MODE", "0") == "1"
SNAPSHOT_DATE = None          # None = latest snapshot on disk; or "YYYY-MM-DD"
SAMPLE_N = None               # e.g. 5 = only the first 5 spots in the demo tables below
SEED = 42
DEMO_WINDOW_HOURS = 48        # window used for the demo table in this notebook

MAX_BUOY_DISTANCE_KM = 60     # [TEAM] decide: nearest buoy is used only if within this distance (and no buoy_station_id is set)
OBS_MAX_AGE_H = 3             # an observation is "current" if at most this old at the reference time
OBS_CONFLICT_RATIO = 0.30     # flag a conflict if observed wave height differs from the forecast by more than 30%
OBS_CORRECTION_ENABLED = True # scale the next hours of forecast towards the observation when they conflict
OBS_CORRECTION_HOURS = 12     # the correction fades out linearly over this many hours

## 1. Thresholds: one editable table
All rule parameters live in `surfability_thresholds.csv`: one row per parameter, one column per skill level, plus `unit`, `description` and `source`. Keeping them in a table (not in code) means:
- the surf-school values can be entered without touching code,
- the table can be copied directly into the report's appendix,
- a sensitivity analysis is just a loop over edited copies of the table.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SKILLS = ["beginner", "intermediate", "advanced"]


def load_thresholds(path=THRESHOLDS_FILE):
    table = pd.read_csv(path).set_index("parameter")
    return {skill: table[skill].astype(float).to_dict() for skill in SKILLS}, table


THRESHOLDS, THRESHOLD_TABLE = load_thresholds()
if not LIBRARY_MODE:
    n_placeholder = THRESHOLD_TABLE["source"].fillna("").str.contains("PLACEHOLDER").sum()
    print(f"{len(THRESHOLD_TABLE)} parameters; {n_placeholder} still marked PLACEHOLDER")
    display(THRESHOLD_TABLE)

## 2. Loading snapshots, spots and observations
A snapshot is loaded once and cached in memory; the search engine calls these functions for every query, so disk access must not happen per query. The *reference time* of a snapshot is the moment it was fetched. Relative expressions such as "tomorrow" are interpreted from that moment, which is what makes old snapshots replayable.

In [ ]:
_SNAPSHOT_CACHE = {}


def list_snapshots():
    if not SNAP_ROOT.exists():
        return []
    return sorted(p.name for p in SNAP_ROOT.iterdir()
                  if p.is_dir() and not p.name.endswith(".tmp") and (p / "manifest.json").exists())


def resolve_snapshot_date(date=None):
    available = list_snapshots()
    if not available:
        raise FileNotFoundError("No snapshots in data/snapshots/. Run 00_data_collection.ipynb first.")
    if date in (None, "", "latest"):
        return available[-1]
    if str(date) not in available:
        raise FileNotFoundError(f"Snapshot {date} not found. Available: {available}")
    return str(date)


def snapshot_from_frames(marine, wind, obs, meta, key):
    """Package in-memory data (e.g. a live fetch) the same way as a saved snapshot."""
    for df in (marine, wind):
        df["time"] = pd.to_datetime(df["time"])
    return {"key": key, "date": meta["fetched_at"][:10], "marine": marine, "wind": wind,
            "obs": obs if obs is not None else pd.DataFrame(), "meta": meta,
            "ref_time": pd.Timestamp(meta["fetched_at"]).floor("h")}


def load_snapshot(date=None):
    date = resolve_snapshot_date(date)
    if date not in _SNAPSHOT_CACHE:
        folder = SNAP_ROOT / date
        obs_file = folder / "cwa_obs.parquet"
        _SNAPSHOT_CACHE[date] = snapshot_from_frames(
            pd.read_parquet(folder / "marine.parquet"), pd.read_parquet(folder / "wind.parquet"),
            pd.read_parquet(obs_file) if obs_file.exists() else pd.DataFrame(),
            json.loads((folder / "manifest.json").read_text(encoding="utf-8")), key=date)
    return _SNAPSHOT_CACHE[date]


def load_spots(path=CORPUS_FILE):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 01_spot_corpus.ipynb first.")
    return {s["spot_id"]: s for s in json.loads(path.read_text(encoding="utf-8"))}


_ALL_OBS = None


def load_all_observations(refresh=False):
    """All CWA observations from every snapshot, de-duplicated (each snapshot holds the previous 48 h)."""
    global _ALL_OBS
    if _ALL_OBS is None or refresh:
        frames = [pd.read_parquet(SNAP_ROOT / d / "cwa_obs.parquet") for d in list_snapshots()
                  if (SNAP_ROOT / d / "cwa_obs.parquet").exists()]
        _ALL_OBS = (pd.concat(frames).drop_duplicates(["station_id", "obs_time"]).sort_values("obs_time")
                    if frames else pd.DataFrame())
    return _ALL_OBS


SPOTS = load_spots()

### Which buoy checks which spot?
The team can set `buoy_station_id` per spot. Otherwise we use the nearest buoy from the station lookup, but only within `MAX_BUOY_DISTANCE_KM`. A buoy far away, or on the other side of the island, measures a different sea, and a wrong "check" is worse than none.

In [ ]:
def _haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def _station_lookup():
    if not STATION_LOOKUP_FILE.exists():
        return pd.DataFrame(columns=["station_id", "station_name_en", "station_type", "lat", "lon"])
    return pd.read_csv(STATION_LOOKUP_FILE, dtype={"station_id": str})


STATIONS = _station_lookup()


def buoy_for_spot(spot):
    """Returns (station_id, english_name) or (None, None)."""
    sid = (spot.get("buoy_station_id") or "").strip()
    if sid:
        match = STATIONS.loc[STATIONS["station_id"] == sid, "station_name_en"]
        return sid, (match.iloc[0] if len(match) else sid)
    buoys = STATIONS[(STATIONS["station_type"] == "buoy") & STATIONS["lat"].notna()]
    if buoys.empty:
        return None, None
    dist = _haversine_km(spot["lat"], spot["lon"], buoys["lat"].astype(float), buoys["lon"].astype(float))
    i = int(np.argmin(dist))
    if dist.iloc[i] > MAX_BUOY_DISTANCE_KM:
        return None, None
    return buoys["station_id"].iloc[i], buoys["station_name_en"].iloc[i]

## 3. One hourly "conditions" frame per spot
Forecasts and observations are converted to the same columns, so the *same* scoring code can score both. That is what lets `05_evaluation` compare forecast-based calls with observation-based calls.

| Column | Meaning |
|---|---|
| `wave_height_m` | significant wave height (all wave components combined) |
| `dir_deg`, `period_s` | direction (*from*) and period of the **dominant** component: swell if the swell is at least as high as the wind sea, else wind sea |
| `wind_kmh`, `wind_dir_deg` | wind speed and direction (*from*) |
| `sea_level_m` | sea level incl. tide → converted to a tide stage (low/mid/high) |

In [ ]:
def _tide_stage(sea_level):
    """Position of each hour between the local low and high water (rolling 25 h window): 0 = low, 1 = high."""
    lo = sea_level.rolling(25, center=True, min_periods=6).min()
    hi = sea_level.rolling(25, center=True, min_periods=6).max()
    return ((sea_level - lo) / (hi - lo)).clip(0, 1)


_FRAME_CACHE = {}


def forecast_frame(spot_id, snapshot):
    """Cached per (snapshot, spot): the search engine asks for the same frame for every query."""
    key = (snapshot["key"], spot_id)
    if key not in _FRAME_CACHE:
        _FRAME_CACHE[key] = _build_forecast_frame(spot_id, snapshot)
    return _FRAME_CACHE[key]


def _build_forecast_frame(spot_id, snapshot):
    m = snapshot["marine"]
    m = m[m["spot_id"] == spot_id].set_index("time").sort_index()
    w = snapshot["wind"]
    w = w[w["spot_id"] == spot_id].set_index("time").sort_index()
    if m.empty:
        return pd.DataFrame()
    swell_dominant = m["swell_wave_height"].fillna(0) >= m["wind_wave_height"].fillna(0)
    f = pd.DataFrame(index=m.index)
    f["wave_height_m"] = m["wave_height"]
    f["dir_deg"] = np.where(swell_dominant, m["swell_wave_direction"], m["wind_wave_direction"])
    f["period_s"] = np.where(swell_dominant, m["swell_wave_period"], m["wind_wave_period"])
    f["dir_deg"] = f["dir_deg"].fillna(m["wave_direction"])
    f["period_s"] = f["period_s"].fillna(m["wave_period"])
    f["swell_dominant"] = swell_dominant
    f["wind_kmh"] = w["wind_speed_10m"].reindex(f.index)
    f["wind_dir_deg"] = w["wind_direction_10m"].reindex(f.index)
    f["sea_level_m"] = m["sea_level_height_msl"]
    f["tide_stage"] = _tide_stage(f["sea_level_m"])
    f["source"] = "forecast"
    return f


def observed_frame(spot_id, snapshot, obs=None):
    """Same columns from CWA observations (buoy waves, buoy wind if present, tide station if mapped).
    Gaps in observed wind or sea level are filled from the forecast and marked in `source`."""
    spot = SPOTS[spot_id]
    obs = load_all_observations() if obs is None else obs
    buoy_id, _ = buoy_for_spot(spot)
    fc = forecast_frame(spot_id, snapshot)
    if obs.empty or buoy_id is None or fc.empty:
        return pd.DataFrame()
    b = obs[obs["station_id"] == buoy_id].copy()
    if b.empty:
        return pd.DataFrame()
    b["time"] = b["obs_time"].dt.floor("h")
    b = b.groupby("time").last()
    f = pd.DataFrame(index=b.index)
    f["wave_height_m"] = b["wave_height_m"]
    f["dir_deg"] = b["wave_dir_deg"]
    f["period_s"] = b["wave_period_s"]
    f["swell_dominant"] = True
    f["wind_kmh"] = (b["wind_speed_ms"] * 3.6).fillna(fc["wind_kmh"].reindex(f.index))
    f["wind_dir_deg"] = b["wind_dir_deg"].fillna(fc["wind_dir_deg"].reindex(f.index))
    tide_id = (spot.get("tide_station_id") or "").strip()
    tide = obs[obs["station_id"] == tide_id].assign(time=lambda d: d["obs_time"].dt.floor("h")).groupby("time").last() if tide_id else pd.DataFrame()
    f["sea_level_m"] = tide["tide_height_m"].reindex(f.index) if not tide.empty else fc["sea_level_m"].reindex(f.index)
    f["tide_stage"] = _tide_stage(f["sea_level_m"])
    f["source"] = "observed"
    return f.dropna(subset=["wave_height_m"])

## 4. Factor scores
Each factor maps one physical quantity to `[0, 1]`:

| Factor | Rule (parameters from the threshold table) |
|---|---|
| **size** | trapezoid: 0 below `wave_min_m`, rising to 1 at `wave_ideal_low_m`, 1 up to `wave_ideal_high_m`, falling to 0 at `wave_max_m` (capped by the spot's `spot_max_wave_m`) |
| **period** | `period_floor_fit` at or below `period_min_s`, rising linearly to 1 at `period_ideal_s` |
| **direction** | 1 inside the spot's swell window, falling to 0 at `swell_window_margin_deg` outside it |
| **wind** | 1 if calm (≤ `wind_glassy_max_kmh`) or moderately offshore; onshore wind lowers the score in proportion to its onshore component and speed |
| **tide** | 1 if the tide stage matches `ideal_tide`, else `tide_mismatch_fit` |
| **skill** | 1 if the spot lists the user's level, else `spot_skill_mismatch_fit` (e.g. 0 for beginners at an advanced-only spot) |

**Why multiply instead of add?** `CF = size^w1 · period^w2 · direction^w3 · wind^w4 · tide^w5 · skill`. With a product, one disqualifying factor (flat, dangerous, swell blocked) makes the whole score low. With a weighted sum, perfect offshore wind could "compensate" for having no waves, which no surfer would accept. The exponents `w` let the team make a factor more or less decisive.

In [ ]:
def angular_diff(a, b):
    return np.abs((np.asarray(a, dtype=float) - b + 180) % 360 - 180)


def outside_window_deg(direction, lo, hi):
    """Degrees by which `direction` lies outside the clockwise window lo -> hi (0 if inside)."""
    direction = np.asarray(direction, dtype=float)
    width = (hi - lo) % 360
    off = (direction - lo) % 360
    return np.where(off <= width, 0.0, np.minimum(off - width, 360 - off))


def trapezoid(x, a, b, c, d):
    xs = np.maximum.accumulate([a, b, c, d])            # guard against inconsistent thresholds
    return np.interp(np.asarray(x, dtype=float), xs, [0, 1, 1, 0], left=0.0, right=0.0)


def tide_label(stage):
    return np.select([stage < 1 / 3, stage > 2 / 3], ["low", "high"], "mid").astype(object)


def wind_label(speed, offshore_cos, glassy_max):
    return np.select([speed <= glassy_max, offshore_cos >= 0.5, offshore_cos <= -0.5],
                     ["light", "offshore", "onshore"], "cross-shore").astype(object)


def factor_scores(frame, spot, skill, th=None):
    t = (th or THRESHOLDS)[skill]
    out = frame.copy()
    spot_max = spot.get("spot_max_wave_m") or math.inf
    wave_max = min(t["wave_max_m"], spot_max)
    out["f_size"] = trapezoid(out["wave_height_m"], t["wave_min_m"], t["wave_ideal_low_m"],
                              min(t["wave_ideal_high_m"], wave_max), wave_max)
    out["f_period"] = np.interp(out["period_s"].fillna(0), [t["period_min_s"], t["period_ideal_s"]],
                                [t["period_floor_fit"], 1.0])
    out["f_direction"] = np.clip(1 - outside_window_deg(out["dir_deg"].fillna(-999) % 360, spot["swell_dir_min_deg"],
                                                        spot["swell_dir_max_deg"]) / t["swell_window_margin_deg"], 0, 1)
    out.loc[out["dir_deg"].isna(), "f_direction"] = 0.5          # unknown direction: neutral-low
    offshore_from = (spot["orientation_deg"] + 180) % 360
    cos_off = np.cos(np.radians(angular_diff(out["wind_dir_deg"].fillna(offshore_from), offshore_from)))
    speed = out["wind_kmh"].fillna(0)
    onshore_weight = np.clip((0.5 - cos_off) / 1.5, 0, 1)        # 0 for offshore/cross-off, 1 for straight onshore
    onshore_pen = np.clip((speed - t["wind_glassy_max_kmh"]) / (t["wind_onshore_max_kmh"] - t["wind_glassy_max_kmh"]), 0, 1)
    too_strong_off = np.clip((speed - t["wind_offshore_max_kmh"]) /
                             (t["wind_offshore_limit_kmh"] - t["wind_offshore_max_kmh"]), 0, 1) * (cos_off >= 0.5)
    out["f_wind"] = np.clip(1 - onshore_weight * onshore_pen - too_strong_off, 0, 1)
    out["wind_label"] = wind_label(speed, cos_off, t["wind_glassy_max_kmh"])
    ideal = set(spot.get("ideal_tide") or ["any"])
    out["tide_label"] = tide_label(out["tide_stage"].fillna(0.5))
    out["f_tide"] = np.where(("any" in ideal) | out["tide_stage"].isna() | out["tide_label"].isin(ideal),
                             1.0, t["tide_mismatch_fit"])
    f_skill = 1.0 if skill in (spot.get("skill_levels") or []) else t["spot_skill_mismatch_fit"]
    out["f_skill"] = f_skill
    out["cf"] = (out["f_size"] ** t["weight_size"] * out["f_period"] ** t["weight_period"]
                 * out["f_direction"] ** t["weight_direction"] * out["f_wind"] ** t["weight_wind"]
                 * out["f_tide"] ** t["weight_tide"] * f_skill)
    out["too_big"] = out["wave_height_m"] > wave_max
    out["flat"] = out["wave_height_m"] < t["wave_min_m"]
    return out

## 5. Forecast vs. current observation
If the buoy that checks a spot reported within the last `OBS_MAX_AGE_H` hours, we compare it with the forecast for that hour. A difference above `OBS_CONFLICT_RATIO` is shown to the user ("conditions uncertain"). With `OBS_CORRECTION_ENABLED`, the forecast wave height for the next `OBS_CORRECTION_HOURS` is scaled towards the observation, fading out linearly. Observations are most informative for the next few hours and say little about the day after tomorrow.

In [ ]:
def current_observation(spot, snapshot, ref_time):
    buoy_id, buoy_name = buoy_for_spot(spot)
    obs = snapshot.get("obs", pd.DataFrame())
    if buoy_id is None or obs.empty:
        return None
    b = obs[(obs["station_id"] == buoy_id) & (obs["obs_time"] <= ref_time)
            & (obs["obs_time"] >= ref_time - pd.Timedelta(hours=OBS_MAX_AGE_H))].dropna(subset=["wave_height_m"])
    if b.empty:
        return None
    row = b.sort_values("obs_time").iloc[-1]
    return {"station_id": buoy_id, "station_name": buoy_name, "time": row["obs_time"], "wave_height_m": float(row["wave_height_m"])}


def apply_observation(frame, observation, ref_time):
    """Returns (frame, note, conflict) with the forecast scaled towards a conflicting observation."""
    if observation is None or frame.empty:
        return frame, None, False
    hour = observation["time"].floor("h")
    if hour not in frame.index or pd.isna(frame.at[hour, "wave_height_m"]) or frame.at[hour, "wave_height_m"] <= 0:
        return frame, None, False
    fc = float(frame.at[hour, "wave_height_m"])
    ratio = observation["wave_height_m"] / fc
    note = (f"Buoy {observation['station_name']} measured {observation['wave_height_m']:.1f} m at "
            f"{observation['time']:%H:%M} (forecast {fc:.1f} m)")
    conflict = abs(ratio - 1) > OBS_CONFLICT_RATIO
    if conflict and OBS_CORRECTION_ENABLED:
        frame = frame.copy()
        hours_ahead = (frame.index - ref_time) / pd.Timedelta(hours=1)
        fade = np.clip(1 - hours_ahead / OBS_CORRECTION_HOURS, 0, 1) * (hours_ahead >= 0)
        frame["wave_height_m"] = frame["wave_height_m"] * (1 + (ratio - 1) * fade)
        note += "; forecast adjusted for the next hours"
    return frame, note, conflict

## 6. Condition fit for a time window, and the explanation
The score of a spot for a window is the score of its **best daylight hour** in that window: a surfer only needs one good session. We also report *all* surfable hours as ranges ("06:00–10:00"), because "when" is half of the user's question.

**Why template explanations (not a language model)?** Every number in the sentence comes straight from the scored frame, so an explanation can never claim conditions that the system did not compute. That *faithfulness* is easy to verify and defend.

In [ ]:
COMPASS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
FACTOR_PHRASES = {"f_size": "wave size", "f_period": "short period", "f_direction": "swell direction",
                  "f_wind": "wind", "f_tide": "tide", "f_skill": "skill level of the spot"}
SKILL_PLURAL = {"beginner": "beginners", "intermediate": "intermediates", "advanced": "advanced surfers"}


def compass(deg):
    return "?" if deg is None or pd.isna(deg) else COMPASS[int((deg % 360) / 22.5 + 0.5) % 16]


def hour_ranges(times):
    """[06:00, 07:00, 08:00, 13:00] -> '06:00-09:00, 13:00-14:00' (end = end of the last good hour)."""
    if len(times) == 0:
        return ""
    times = sorted(times)
    ranges, start, prev = [], times[0], times[0]
    for t in times[1:]:
        if t - prev != pd.Timedelta(hours=1):
            ranges.append((start, prev))
            start = t
        prev = t
    ranges.append((start, prev))
    fmt = lambda a, b: (f"{a:%a %H:%M}-{b + pd.Timedelta(hours=1):%H:%M}")
    return ", ".join(fmt(a, b) for a, b in ranges)


def verdict(cf, t):
    if cf >= t["good_threshold"]:
        return "Good"
    if cf >= t["surfable_threshold"]:
        return "Fair"
    return "Poor"


def build_explanation(spot, best, skill, surf_hours, t, obs_note=None, source="forecast"):
    if best is None:
        return "No forecast data for the requested time."
    v = verdict(best["cf"], t)
    parts = [f"{v} for {SKILL_PLURAL[skill]}."]
    swell_word = "swell" if best["swell_dominant"] else "wind swell"
    parts.append(f"Best around {best.name:%a %H:%M}: {best['wave_height_m']:.1f} m {compass(best['dir_deg'])} "
                 f"{swell_word} at {best['period_s']:.0f} s, wind {best['wind_kmh']:.0f} km/h "
                 f"{compass(best['wind_dir_deg'])} ({best['wind_label']}), {best['tide_label']} tide.")
    if surf_hours:
        parts.append(f"Surfable {surf_hours}.")
    if best["too_big"]:
        parts.append(f"Warning: too big for {SKILL_PLURAL[skill]} (max {min(t['wave_max_m'], spot.get('spot_max_wave_m') or math.inf):.1f} m).")
    elif best["flat"]:
        parts.append("Too small: flat for this level.")
    elif v != "Good":
        limiting = min(FACTOR_PHRASES, key=lambda f: best[f])
        parts.append(f"Limited by {FACTOR_PHRASES[limiting]}.")
    if source == "observed":
        parts.append("(Scored from buoy observations.)")
    if obs_note:
        parts.append(obs_note + ".")
    return " ".join(parts)


def condition_fit(spot_id, snapshot, skill, start, end, ref_time=None, use_observations=False, th=None):
    """Score one spot for one skill level and time window [start, end). Returns a dict (see keys below)."""
    th = th or THRESHOLDS
    t = th[skill]
    spot = SPOTS[spot_id]
    ref_time = snapshot["ref_time"] if ref_time is None else pd.Timestamp(ref_time)
    frame = observed_frame(spot_id, snapshot) if use_observations else pd.DataFrame()
    source = "observed" if not frame.empty else "forecast"
    obs_note, conflict = None, False
    if frame.empty:
        frame = forecast_frame(spot_id, snapshot)
        frame, obs_note, conflict = apply_observation(frame, current_observation(spot, snapshot, ref_time), ref_time)
    if frame.empty:          # spot not in this snapshot (e.g. added to the corpus after the snapshot was taken)
        return {"spot_id": spot_id, "cf": 0.0, "surfable": False, "best_time": None, "verdict": "No data",
                "surfable_hours": "", "explanation": "No forecast for this spot in this snapshot.",
                "too_big": False, "conflict": False, "source": source, "factors": {}}
    window = frame[(frame.index >= pd.Timestamp(start)) & (frame.index < pd.Timestamp(end))
                   & (frame.index.hour >= t["first_light_hour"]) & (frame.index.hour < t["last_light_hour"])]
    if window.empty:
        return {"spot_id": spot_id, "cf": 0.0, "surfable": False, "best_time": None, "verdict": "No data",
                "surfable_hours": "", "explanation": build_explanation(spot, None, skill, "", t),
                "too_big": False, "conflict": conflict, "source": source, "factors": {}}
    scored = factor_scores(window, spot, skill, th)
    best = scored.loc[scored["cf"].idxmax()]
    surf_hours = hour_ranges(list(scored.index[scored["cf"] >= t["surfable_threshold"]]))
    return {
        "spot_id": spot_id, "cf": float(best["cf"]), "surfable": bool(best["cf"] >= t["surfable_threshold"]),
        "best_time": best.name, "verdict": verdict(best["cf"], t), "surfable_hours": surf_hours,
        "explanation": build_explanation(spot, best, skill, surf_hours, t, obs_note, source),
        "too_big": bool(scored["too_big"].all()), "conflict": conflict, "source": source,
        "factors": {f: round(float(best[f]), 3) for f in FACTOR_PHRASES},
        "wave_height_m": float(best["wave_height_m"]),
        "conditions": {"wave_height_m": float(best["wave_height_m"]), "period_s": float(best["period_s"]),
                       "dir_deg": float(best["dir_deg"]), "wind_kmh": float(best["wind_kmh"]),
                       "wind_dir_deg": float(best["wind_dir_deg"]), "wind_label": best["wind_label"],
                       "tide_label": best["tide_label"], "swell_dominant": bool(best["swell_dominant"])},
        "weights": {f: t[w] for f, w in [("f_size", "weight_size"), ("f_period", "weight_period"),
                                         ("f_direction", "weight_direction"), ("f_wind", "weight_wind"),
                                         ("f_tide", "weight_tide")]},
    }


def daily_outlook(spot_id, snapshot, skill, th=None):
    """Best condition-fit score per day (for the 7-day timeline)."""
    th = th or THRESHOLDS
    frame = forecast_frame(spot_id, snapshot)
    t = th[skill]
    if frame.empty:
        return pd.Series(dtype=float)
    frame = frame[(frame.index.hour >= t["first_light_hour"]) & (frame.index.hour < t["last_light_hour"])]
    if frame.empty:
        return pd.Series(dtype=float)
    scored = factor_scores(frame, SPOTS[spot_id], skill, th)
    return scored["cf"].groupby(scored.index.date).max()

## 7. Try it on the latest snapshot
A table of all spots × skill levels for the next 48 hours, then a heatmap of the best daily score per spot. The heatmap is the "7-day surfability timeline" that also appears in the demo.

In [ ]:
if not LIBRARY_MODE:
    snap = load_snapshot(SNAPSHOT_DATE)
    ref = snap["ref_time"]
    print(f"Snapshot {snap['date']} (fetched {snap['meta']['fetched_at']}, status {snap['meta']['status']})")
    spot_ids = list(SPOTS)[:SAMPLE_N] if SAMPLE_N else list(SPOTS)
    rows = []
    for sid in spot_ids:
        for skill in SKILLS:
            r = condition_fit(sid, snap, skill, ref, ref + pd.Timedelta(hours=DEMO_WINDOW_HOURS))
            rows.append({"spot": SPOTS[sid]["name"], "skill": skill, "cf": round(r["cf"], 2), "verdict": r["verdict"],
                         "best_time": r["best_time"], "surfable_hours": r["surfable_hours"]})
    table = pd.DataFrame(rows)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(RESULTS_DIR / "surfability_table.csv", index=False)
    display(table)

In [ ]:
if not LIBRARY_MODE:
    example = condition_fit(spot_ids[0], snap, "intermediate", ref, ref + pd.Timedelta(hours=DEMO_WINDOW_HOURS))
    print(SPOTS[spot_ids[0]]["name"], "->", example["explanation"])
    print("Factor scores at the best hour:", example["factors"])

In [ ]:
if not LIBRARY_MODE:
    heat = pd.DataFrame({SPOTS[sid]["name"]: daily_outlook(sid, snap, "intermediate") for sid in spot_ids}).T
    fig, ax = plt.subplots(figsize=(1.2 * max(len(heat.columns), 4) + 2, 0.45 * len(heat) + 1.5))
    im = ax.imshow(heat.values, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(heat.columns)), [pd.Timestamp(d).strftime("%a %d") for d in heat.columns])
    ax.set_yticks(range(len(heat)), heat.index)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="best condition-fit score (intermediate)")
    ax.set_title(f"7-day surfability outlook, snapshot {snap['date']}")
    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "surfability_heatmap.png", dpi=200)
    plt.show()